# 🏋️ Module 7.5: Exercises

Practice deployment concepts.

---

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
import requests
import json
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

mlflow.autolog(disable=True)
mlflow.set_experiment("07_exercises")

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("✅ Ready!")

## Exercise 1: Train, Log, and Serve

1. Train a `GradientBoostingClassifier` with `n_estimators=200, learning_rate=0.05`
2. Log it with a signature and input example
3. Print the `mlflow models serve` command to serve it on port **5001**
4. (Optional) Start the server in a terminal and send a prediction from this notebook

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION
with mlflow.start_run(run_name="ex1_serve_model"):
    model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42)
    model.fit(X_train, y_train)
    
    sig = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(model, "model", signature=sig, input_example=X_train[:2])
    mlflow.log_metric("accuracy", accuracy_score(y_test, model.predict(X_test)))
    
    rid = mlflow.active_run().info.run_id

print(f"Run ID: {rid}")
print(f"\n📋 Serve command:")
print(f"mlflow models serve -m runs:/{rid}/model -p 5001 --no-conda")

## Exercise 2: Build a Prediction Client

Write a function `predict_via_api(data, port=1234)` that:
1. Formats the data as `dataframe_split` JSON
2. Sends a POST request to the model server
3. Returns the predictions
4. Handles connection errors gracefully

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION
def predict_via_api(data, port=1234):
    """Send prediction request to MLFlow model server."""
    url = f"http://127.0.0.1:{port}/invocations"
    payload = {
        "dataframe_split": {
            "columns": list(data.columns),
            "data": data.values.tolist()
        }
    }
    try:
        response = requests.post(
            url, headers={"Content-Type": "application/json"}, json=payload
        )
        response.raise_for_status()
        return response.json()
    except requests.ConnectionError:
        return {"error": f"Server not running on port {port}"}
    except requests.HTTPError as e:
        return {"error": str(e)}

# Test it (will show error if server isn't running — that's OK)
result = predict_via_api(X_test[:3], port=1234)
print(f"Result: {result}")

## Exercise 3: Docker Build Command

Write out the complete sequence of terminal commands to:
1. Build a Docker image from the model you logged in Exercise 1
2. Run the container on port 9090
3. Test it with curl
4. Stop the container

In [ ]:
# YOUR CODE HERE — print the commands


In [ ]:
# ✅ SOLUTION
print("📋 Docker deployment commands:")
print(f"")
print(f"# 1. Build Docker image")
print(f"mlflow models build-docker -m runs:/{rid}/model -n gb-wine-model")
print(f"")
print(f"# 2. Run container on port 9090")
print(f"docker run -p 9090:8080 gb-wine-model")
print(f"")
print(f"# 3. Test with curl")
print(f'curl -X POST http://localhost:9090/invocations -H "Content-Type: application/json" -d \'{{"dataframe_split": {{"columns": {list(X_test.columns[:3])}, "data": [[1.0, 2.0, 3.0]]}}}}\'')
print(f"")
print(f"# 4. Stop container")
print(f"docker ps  # Find container ID")
print(f"docker stop <container_id>")

---

## 🎓 Module 7 Complete!

**Next up**: Module 8 — Advanced Topics →